## Format from Stanza (conllu) to Pos (AntConc Pos format)
### the normalised version of the token is added under ‘n=’

In [9]:
import os
import time
import unicodedata
from stanza.utils.conll import CoNLL

def replace_special_chars(text: str) -> str:
    """
    Replaces specific old Cyrillic characters with their modern equivalents.
    """
    replacements = {
        "ꙋ": "у",
        "ѹ": "у",
        "ѿ": "от",
        "Ѿ": "От",
        "ѡ": "о",
        "Ѡ": "О",
        "ѕ": "з",
        "ꙁ": "з",
        "ѻ": "о",
        "ꙍ": "о",
        "ѧ": "я",
        "ꙗ": "я",
        "Ꙗ": "Я",
        "\u1C83": "с",  # CYRILLIC SMALL LETTER WIDE ES
        "\u1C84": "т",  # CYRILLIC SMALL LETTER TALL TE
    }
    
    for old_char, new_char in replacements.items():
        text = text.replace(old_char, new_char)
    
    return text


def remove_polytonic_chars(text: str) -> str:
    """
    Removes diacritical marks from polytonic Greek, returning only base letters.
    """
    normalized_text = unicodedata.normalize('NFD', text)
    return ''.join(c for c in normalized_text if not unicodedata.combining(c) and c not in "῾᾿")
    
def convert_conllu_to_pos(input_path, pos_path):
    """
    Reads a CoNLL-U formatted file and converts it to a simplified POS-tagged format.
    """
    with open(input_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    pos_text = ""
    sentence = []
    
    for line in lines:
        line = line.strip()
        if line and not line.startswith("#"):
            columns = line.split("\t")
            if len(columns) > 3:
                word_text = columns[1]  # Token
                lemma = columns[2]  # Lemma
                upos = columns[3]  # Universal POS Tag
                feats = columns[5]  # Feats
                
                nword = replace_special_chars(word_text)
                nword = remove_polytonic_chars(nword)
                
                if upos == "PUNCT":
                    sentence.append(f"{word_text}")
                else:
                    sentence.append(f" {word_text}_{upos}|l={lemma}|n={nword}|{feats}")
        else:
            if sentence:
                pos_text += "".join(sentence) + "\n"
                sentence = []
    
    with open(pos_path, 'w', encoding='utf-8') as file:
        file.write(pos_text)

def process_directory(input_dir, output_dir):
    """
    Iterates through the directory and converts all CoNLL-U files to POS format.
    """
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".conllu"):
                input_path = os.path.join(root, file)
                relative_path = os.path.relpath(input_path, os.path.join(input_dir, 'conllu'))
                pos_path = os.path.join(output_dir, relative_path.replace(".conllu", ".txt"))
                
                os.makedirs(os.path.dirname(pos_path), exist_ok=True)
                
                start_time = time.time()
                convert_conllu_to_pos(input_path, pos_path)
                end_time = time.time()
                print(f"File '{input_path}' converted to '{pos_path}' in {end_time - start_time:.2f} seconds.")

# Main execution
if __name__ == "__main__":
    input_directory = "./test-chu-out"
    output_directory = os.path.join(input_directory, 'pos')
    
    process_directory(input_directory, output_directory)

File './test-chu-out\conllu\txt\PPN1023248611_combined_output.conllu' converted to './test-chu-out\pos\txt\PPN1023248611_combined_output.txt' in 0.05 seconds.
